<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    08 · Ciencia de datos: SVM y Random Forest
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:620px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 4 — Sin experiencia previa en programación</em>
  </p>
</div>

# Ciencia de Datos — Predicciones Toxicológicas

---
### En esta lección aprenderás:

- cómo entrenar una SVM.
- por qué es necesario escalar las variables.
- cómo entrenar un modelo de Random Forest.
- sobre el Y-scrambling y la necesidad de dividir los datos en entrenamiento y prueba.
- cómo dividir los datos en un conjunto de entrenamiento y uno de prueba.

---

In [ ]:
import pandas as pd 
import numpy as np
!pip install rdkit==2022.3.4
!pip install searborn
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import AllChem as Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from rdkit.Chem.Lipinski import * 
from rdkit.Chem.rdMolDescriptors import CalcExactMolWt, CalcTPSA
from rdkit.Chem.Crippen import MolLogP, MolMR
from sklearn.preprocessing import MinMaxScaler
import seaborn as  sns
from matplotlib import pyplot as plt
import sys
%matplotlib inline

if 'google.colab' in sys.modules: # checks whether the notebook runs on collab
  !wget https://raw.githubusercontent.com/kochgroup/intro_pharma_ai/main/utils/utils.py
  %run utils.py
else:
  %run ../utils/utils.py # loads prewritten function

# Escalado

El escalado de variables puede ser crítico para el éxito de los modelos de machine learning. Escalar variables significa que cambiamos la escala de los valores de una variable. Más específicamente, queremos que todas las variables usen la misma escala.

Por ejemplo, el valor de una casa en € generalmente oscila entre 50.000 y 1.000.000. En cambio, el número de habitaciones oscila entre 1 y 10. Si se introducen estas dos variables sin escalar en un modelo, el modelo puede sobreponderar el precio de la casa simplemente porque tiene valores mucho mayores. Esto se puede evitar escalando las variables.

El escalado **min-max** es una de las técnicas de escalado más simples y comunes. Transforma los valores de modo que el mínimo sea 0 y el máximo sea 1. Todos los demás valores se ubican proporcionalmente entre 0 y 1.

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$


Primero, carga los datos usando Python.

___
**Tipos de datos:**
<br> <br>
Existen muchos formatos diferentes en los que se almacenan los datos. Probablemente el más común es el formato `comma separated value`. Este se puede reconocer por la abreviatura `.csv` al final de un archivo. Sin embargo, los datos también se pueden almacenar separados por otros caracteres. En el siguiente ejemplo se usa una tabulación `\t` como separador. <div style="float: right;"><i>Los archivos separados por tabulaciones también se suelen guardar con la extensión `.tab` o `.tsv`.</i></div>

In [ ]:
print("ab")
print("a\tb")

Como ya habrás notado en algunas de las librerías cargadas, hoy usaremos algunas funciones que aún no conoces.

- `pandas` contiene funciones útiles para procesar grandes cantidades de datos y puede hacer prácticamente todo lo que hace Excel (y algunas cosas incluso mejor). `pandas` crea una estructura de datos llamada `DataFrame`, que se parece a una tabla de Excel.
- `numpy` es una librería para cálculos numéricos en Python.
- `sklearn` (también llamado `scikit-learn`) es la librería de machine learning más popular de Python.
- `matplotlib` es una librería para crear gráficos.

Hoy también usaremos funciones que ya hemos definido para ti. Estas se encuentran en el archivo `utils.py`. Con `from utils import *` cargamos todas las funciones de ese archivo.

In [ ]:
toy_example = pd.read_csv("https://uni-muenster.sciebo.de/s/Xxk3Q1zPIfjYMPz/download", sep = "\t")

print("Type:", type(toy_example))
print("Shape:",toy_example.shape)
toy_example.head()


Los datos leídos con `pandas` se almacenan en un `DataFrame`. Es una tabla con filas y columnas. Con `.head()` puedes mostrar las primeras 5 filas de un `DataFrame`. Los datos contienen tres columnas: `x1`, `x2` e `y`. En total el archivo contiene 150 entradas, es decir, 150 puntos de medición.


In [ ]:
plt.plot(toy_example.x1, toy_example.x2,"o")
plt.plot( toy_example.x1[toy_example.y==1],  toy_example.x2[toy_example.y==1],"o")
plt.xlabel("x1")
plt.ylabel("x2")

Como puedes ver, una clase "rodea" a la otra. Queremos entrenar una SVM para distinguir las dos clases.

También debes notar que las escalas de las dos variables de entrada `x1` y `x2` difieren significativamente.

Los valores de `x1` se encuentran aproximadamente entre `-1` y `1`. <br>Los valores de `x2` se encuentran aproximadamente entre `-500` y `500`.

Entrenemos una SVM sin escalar primero para ver qué ocurre.

In [ ]:
from sklearn.svm import SVC

model = SVC()
model.fit(toy_example.iloc[:,:2], toy_example.y)
y_pred = model.predict(toy_example.iloc[:,:2])
y_pred

Para evaluar la calidad de las predicciones, puedes usar nuevamente la exactitud (accuracy). Usa la misma función del notebook anterior. Calcula la exactitud para las predicciones de este modelo.

In [ ]:
def accuracy(y_true, y_pred):
    return np.sum(y_true==y_pred)/len(y_true)

accuracy(_____, ____)

<details>
    <summary><b>Solución:</b></summary>

```python
accuracy(toy_example.y, y_pred)
```
</details>

La exactitud es aproximadamente `0.7`. No está mal, pero aún hay margen de mejora. Con una de las funciones predefinidas en `utils.py` también puedes visualizar los límites de decisión.

In [ ]:
plot_svc(toy_example, model)

Puedes ver que el límite de decisión está alargado. Esto se debe a que la escala de `x2` es mayor. Esto significa que la distancia desde el límite de decisión hasta los puntos de datos de las dos clases (que debe maximizarse) es más fácil de maximizar para `x2` que para `x1`. Por lo tanto, vemos un buen límite de decisión en la dirección de `x2`, pero no en la dirección de `x1`. Cuando las dos variables tienen la misma escala, este problema no existe.

Escala los datos con la función `min_max` definida a continuación.

In [ ]:
def min_max(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))

toy_example.x1 = min_max(toy_example.x1)
toy_example.x2 = min_max(toy_example.x2)

In [ ]:
plt.plot(toy_example.x1, toy_example.x2,"o")
plt.plot( toy_example.x1[toy_example.y==1],  toy_example.x2[toy_example.y==1],"o")
plt.xlabel("x1")
plt.ylabel("x2")

El gráfico se ve exactamente igual que el anterior, pero las escalas — es decir, los ejes x e y — han cambiado. Es decir, la relación relativa entre los valores no ha cambiado.
¿Puedes crear ahora un `model_2` con `SVC` que se entrene con los valores escalados? ¡Calcula también la exactitud!

In [ ]:
model_2 = ____
model_2.fit(_____, ______)
y_pred = model_2.predict(toy_example._____)
accuracy(______,_____)

<details>
    <summary><b>Solución:</b></summary>

```python
model_2 = SVC()
model_2.fit(toy_example.iloc[:,:2], toy_example.y)
y_pred = model_2.predict(toy_example.iloc[:,:2])
accuracy(toy_example.y, y_pred)
```
</details>

El escalado de las variables de entrada por sí solo resulta en una mejora de 0.3 en la exactitud.
También se puede ver una mejora significativa en el límite de decisión.

In [ ]:
plot_svc(toy_example, model_2)

Aunque los datos fueron generados solo para este ejemplo, muestra por qué escalar las variables de entrada es tan importante.

# División en Entrenamiento y Prueba

De un ejemplo teórico pasamos ahora a una tarea práctica: una predicción toxicológica.
`pandas` puede cargar datos automáticamente desde sitios web y guardarlos como un `DataFrame`.

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/filipsPL/tox21_dataset/master/compounds/sr-mmp.tab", sep = "\t")

print("Shape:",data.shape)
data.head()

Los datos contienen tres columnas: `Compound`, `SMILES` y `activity`. En total el archivo contiene 2246 moléculas.

- `Compound` contiene el ID que puede usarse para distinguir los valores en el conjunto de datos original.
- `SMILES` contiene los `strings` de SMILES.
- `activity` contiene los resultados del ensayo biológico. `1` significa que la molécula es activa (tóxica), `0` significa que es inactiva.

In [ ]:
np.sum(data.activity) # data.activity selects the column 'activity' contained in 'data'

La columna `Compounds` no es importante para el análisis. Por lo tanto, la eliminamos de `data`. Después renombramos las columnas para que todos los nombres estén en minúsculas.

In [ ]:
data = data.iloc[:,1:] # all columns except the first one (index 0) are selected
data.columns = ["smiles", "activity"]
data.head()

Tienes los datos, pero no una entrada para tu modelo. Los SMILES son variables `str`, pero los modelos necesitan variables numéricas como entrada. Esto significa que aún debes crear las llamadas **features** (características). Para esto puedes usar los descriptores del notebook de "Quimioinformática". En las siguientes celdas convertimos los SMILES a `mol` con `rdkit` y calculamos algunos descriptores.

¿Puedes completar el código?

In [ ]:
# convertir todos los SMILES a mol
mols = np.array([Chem.MolFromSmiles(x) for x in ________ ]) # ¿qué SMILES hay que elegir?

# 1) Número de donadores de puente de H
num_hb_donors = [NumHDonors(x) for x in _______ ] # ¿sobre qué variables iteramos?

# 2) Aceptores de puente de H
num_hb_acceptors = [NumHAcceptors(x) for x in _______ ]

# 3) LogP
logP = [MolLogP(x) for x in _______ ]

# 4) Peso molecular
mw = [MolWt(x) for x in _______ ]

# 5) Refracción molar
mr = [MolMR(x) for x in _______ ]

# 6) Área de superficie polar topológica
tpsa = [TPSA(x) for x in _______ ]

# 7) Número de enlaces rotables
num_rotable_bonds = [NumRotatableBonds(x) for x in _______ ]

<details>
    <summary><b>Solución:</b></summary>

```python
# convertir todos los SMILES a mol
mols = np.array([Chem.MolFromSmiles(x) for x in data.smiles])

# 1) Número de donadores de puente de H
num_hb_donors = [NumHDonors(x) for x in mols]

# 2) Aceptores de puente de H
num_hb_acceptors = [NumHAcceptors(x) for x in mols]

# 3) LogP
logP = [MolLogP(x) for x in mols]

# 4) Peso molecular
mw = [MolWt(x) for x in mols]

# 5) Refracción molar
mr = [MolMR(x) for x in mols]

# 6) Área de superficie polar topológica
tpsa = [TPSA(x) for x in mols]

# 7) Número de enlaces rotables
num_rotable_bonds = [NumRotatableBonds(x) for x in mols]
```
</details>

El `DataFrame` `aux_data` tiene una columna separada para cada descriptor, siete en total. La primera fila contiene los descriptores para la primera molécula, la segunda fila para la segunda, y así sucesivamente.
Ahora puedes usar este conjunto de datos para construir un modelo.
Sin embargo, se requieren algunos pasos adicionales antes de poder entrenar el modelo.

In [ ]:
# Primero dividimos el conjunto de datos en 'x' e 'y', es decir, entrada y salida
x = aux_data.iloc[:,:7].values # '[:,:7]' => todas las columnas excepto la columna 7
y = aux_data.iloc[:,7].values  # '[:,7]'  => solo la columna 7

La nueva variable `x` es ahora un `np.array` (en lugar de un `DataFrame`). Esto fue posible gracias a la extensión `.values`. Así puedes convertir rápidamente `pd.DataFrame` a `np.arrays`.

Ahora puedes entrenar un modelo de Random Forest. Similar a `SVC`, primero debes crear una variable con el modelo y luego usar `.fit()`. Ya hay un código preparado para ti — ¡solo tienes que ejecutarlo!

In [ ]:
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
# n_estimators define el número de árboles a usar
rf.fit(x, y)

Para ver qué tan bueno es el modelo, necesitamos extraer las predicciones del modelo para nuestros datos. A diferencia de lo usual, usamos la función <br>`.predict_proba(x)[:,1]`. Usando el modelo `rf` entrenado, hacemos las predicciones para `y_hat`.

In [ ]:
y_hat=rf.predict_proba(x)[:,1]
y_hat

La probabilidad predicha para la primera molécula es `0.014`. Al igual que en la regresión logística, esto significa que según nuestro modelo la molécula es activa en el ensayo MMP el 1.4% de las veces, es decir, es poco probable que sea tóxica. Cuanto mayor sea la probabilidad, más probable es (según el modelo) que la molécula sea activa (tóxica).

In [ ]:
y_pred = np.round(y_hat)
accuracy(y, y_pred)

Luego podemos calcular el AUC con la función `roc_auc_score()`.
A diferencia de `accuracy`, aquí se usan las probabilidades `y_hat` en lugar de los valores redondeados `y_pred`.

In [ ]:
roc_auc_score(y, ____)

<details>
    <summary><b>Solución:</b></summary>

```python
roc_auc_score(y, y_hat)
```
</details>

Muy bien. El modelo es casi perfecto para las predicciones. Toma la decisión correcta en el 99% de los casos. En la siguiente celda, las moléculas mal clasificadas se seleccionan y se muestran con su probabilidad predicha. No es necesario que entiendas este código.

## Y-Scrambling

##### El problema:

*¿Aprendió realmente el modelo o simplemente memorizó los datos de entrenamiento?*

Esta pregunta es de fundamental importancia en machine learning. Un modelo que simplemente memoriza los datos de entrenamiento no puede generalizar a nuevos datos.

**El Y-scrambling** es un método popular para comprobar si el modelo realmente aprendió algo. La idea es desordenar aleatoriamente las etiquetas `y` — de modo que ya no correspondan a las variables `x`. Esto elimina cualquier relación real entre los datos de entrada y salida. Si el modelo sigue siendo bueno después del Y-scrambling, entonces no aprendió nada útil — simplemente memorizó los datos.

Sin embargo, hay un problema: si entrenamos el modelo con los datos y también lo evaluamos con los mismos datos, el modelo puede simplemente memorizar los datos de entrenamiento.

Para solucionar esto, existe el concepto de **división en entrenamiento y prueba**. La idea es dividir los datos en dos partes: el conjunto de **entrenamiento** y el conjunto de **prueba**. Solo el conjunto de entrenamiento se usa para entrenar el modelo. El conjunto de prueba se usa para evaluar el modelo. De esta manera, podemos verificar si el modelo realmente aprendió o simplemente memorizó los datos de entrenamiento.

In [ ]:
falsly_classified=np.where(y_pred!=y)[0]
Draw.MolsToGridImage(mols[falsly_classified],
                     legends=["Probabilidad Predicha:\n"+str(np.round(x,3)) for x in y_hat[falsly_classified]],
                     useSVG=True)

It is notable that the probabilities for these molecules are mostly relatively close to 0.5. This means that the model was not very sure about these molecules. However, in total only 19 molecules were misclassified, so we should not worry too much.


## Y-Scrambling

##### The Problem:

*Did the model really learn what is important for the MMP assay. Or did the RF model just memorize our data?*.

We can find this out with a simple test. We train the RF model again, but shuffle the variable `activity` randomly. That is, the true measurements are shuffled and redistributed, randomly, among the molecules. This process is also called **Y-scrambling**. 

Suppose our real data looks like this:

smiles|Deskriptor 1| Deskriptor 2|activity
------|------------|-------------|--------
SMILES 1|$x_{1,1}$ |$x_{1,2}$|$y_1$
SMILES 2|$x_{2,1}$ |$x_{2,2}$|$y_2$
SMILES 3|$x_{3,1}$ |$x_{3,2}$|$y_3$
SMILES 4|$x_{4,1}$ |$x_{4,2}$|$y_4$

Two descriptors were calculated for each SMILES. We also recorded the activity ($y_1$-$y_4$) for each molecule. The activity $y_1$ is the measured activity of SMILES 1 and so on.

After *Y-scrambling*, our data looks like this:

smiles|Deskriptor 1| Deskriptor 2|activity
------|------------|-------------|--------
SMILES 1|$x_{1,1}$ |$x_{1,2}$|$y_2$
SMILES 2|$x_{2,1}$ |$x_{2,2}$|$y_3$
SMILES 3|$x_{3,1}$ |$x_{3,2}$|$y_4$
SMILES 4|$x_{4,1}$ |$x_{4,2}$|$y_1$

The $y$ values were randomly assigned to other molecules.

El Y-scrambling lleva a la pérdida de la relación real entre las variables `x` (es decir, nuestros descriptores como logP,...) y la variable `y` a predecir. Esto es porque la relación ahora es simplemente aleatoria. Si nuestro Random Forest realmente aprende patrones en lugar de memorizar los datos, entonces el Y-scrambling debería resultar en un modelo muy malo. Sin embargo, si el modelo tiene una buena exactitud incluso después del Y-scrambling, entonces no aprendió nada — simplemente memorizó los datos.

Primero entrenamos el modelo con los datos originales y luego con los datos desordenados. Compararemos los resultados.

In [ ]:
import random
random.seed(15) # una semilla asegura que todos obtengan los mismos datos aleatorios
y_random=np.array(y) # primero almacenamos y en un array
random.shuffle(y_random) # luego desordenamos y_random

Acabamos de desordenar aleatoriamente la información de actividad. Ahora podemos entrenar un modelo de Random Forest. Sin embargo, esta vez no usamos `y`, sino `y_random`.

In [ ]:
# re-entrenar el modelo
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(x, ______) # ¿qué variable y necesitas?
y_hat=rf.predict_proba(x)[:,1]

<details>
    <summary><b>Solución:</b></summary>

```python
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(x, y_random)
y_hat=rf.predict_proba(x)[:,1]
```
</details>

Ahora calcula la exactitud nuevamente (asegúrate de usar `y_random` y no `y` para `y_true`):

In [ ]:
y_pred = np.round(y_hat)
accuracy(y_random, y_pred)

En efecto, la exactitud empeora después del Y-scrambling. Pero la exactitud sigue siendo superior al 90%. El modelo de Random Forest sigue siendo relativamente bueno para predecir la toxicidad, aunque los datos ya no tienen ningún sentido. El modelo no pudo haber aprendido nada, ya que no existe ninguna relación entre `x` e `y`. Entonces, ¿por qué el modelo sigue siendo relativamente bueno?

La respuesta es el **sobreajuste** (overfitting): cuando el Random Forest entrena con los mismos datos que luego se usan para evaluar el modelo, puede simplemente memorizar los datos de entrenamiento. Esto se llama sobreajuste. El modelo no generaliza a nuevos datos.

La solución es usar el Y-scrambling **junto con** una división en entrenamiento y prueba. En la siguiente celda dividimos los datos en un conjunto de entrenamiento y uno de prueba.

In [ ]:
train, test=train_test_split(aux_data, test_size=0.2, train_size=0.8, random_state=1234)

train_x = train.iloc[:,:7]
train_y = train.iloc[:,7]
test_x = test.iloc[:,:7]
test_y = test.iloc[:,7]
f"Forma entrenamiento: {train.shape}, Forma prueba: {test.shape}"

El conjunto de entrenamiento contiene 1796 moléculas y el conjunto de prueba 450.

Primero entrenamos el Random Forest solo con los datos de entrenamiento:

In [ ]:
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(train_x, train_y)

Pero solo hacemos las predicciones para las moléculas del conjunto de prueba.

In [ ]:
y_hat=rf.predict_proba(test_x)[:,1]
y_pred = np.round(y_hat)

Ahora también podemos calcular la exactitud. ¿Qué variable necesitamos ahora para `y_true`?

In [ ]:
accuracy(___, y_pred)

<details>
    <summary><b>Solución:</b></summary>

```python
accuracy(test_y, y_pred)
```
</details>

La exactitud ha empeorado bastante, pero sigue siendo buena. Sin embargo, esta vez podemos estar seguros de que el rendimiento no se debe a la memorización, ya que el modelo nunca ha visto estas moléculas. Si ahora aplicamos el Y-scrambling, el rendimiento debería deteriorarse drásticamente.
Reemplazamos la columna `activity` en `aux_data` con los valores desordenados de `y_random`:

In [ ]:
aux_data.activity = y_random 

Luego dividimos los datos nuevamente en conjunto de entrenamiento y prueba.

In [ ]:
train_random, test_random=train_test_split(aux_data,test_size= 0.2, train_size= 0.8, random_state=1234)
train_x_random = train_random.iloc[:,:7]
train_y_random = train_random.iloc[:,7]
test_x_random = test_random.iloc[:,:7]
test_y_random =  test_random.iloc[:,7]

Repetimos el entrenamiento con los datos desordenados. Luego dejamos que el modelo haga las predicciones para el conjunto de prueba.

In [ ]:
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(train_x_random, train_y_random)
y_hat=rf.predict_proba(test_x_random)[:,1]

Finalmente, calculamos la exactitud.

In [ ]:
y_pred = np.round(y_hat)
accuracy(test_y_random, y_pred)

Usando un conjunto de prueba, la exactitud del modelo con los datos Y-scrambled cae drásticamente a aproximadamente el 50%. No es mejor que un modelo que simplemente adivinara.
Solo usando un conjunto de prueba pudimos demostrar que el modelo aprendió algo más allá de la memorización.
El punto principal era mostrar la importancia de evaluar los modelos con datos que no se usaron durante el entrenamiento.

## Importancia de Features

Como paso final, veamos la **Importancia de Features** (Feature Importance). La importancia de features indica qué tan importante es cada variable de entrada para la decisión. Dependiendo del algoritmo de ML que uses, puedes extraer la importancia de features de forma relativamente sencilla. Primero re-entrenamos nuestro RF, esta vez con los datos originales (no desordenados).

In [ ]:
aux_data.activity = data.activity

train, test=train_test_split(aux_data,test_size= 0.2, train_size= 0.8, random_state=1234)
train_x = train.iloc[:,:7]
train_y = train.iloc[:,7]
test_x = test.iloc[:,:7]
test_y =  test.iloc[:,7]


# train model
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(train_x, train_y)
y_hat=rf.predict_proba(test_x)[:,1]


In [ ]:
feat_importances = pd.Series(rf.feature_importances_, index=aux_data.columns.values[:-1])
feat_importances.nlargest(20).nsmallest(20).plot(kind='barh')

Es claramente evidente que el LogP es el parámetro más importante para determinar la toxicidad, mientras que el número de donadores y aceptores de puente de H son menos relevantes.

Que el valor de LogP sea importante no sorprende. Por ejemplo, podemos observar los gráficos de densidad de las moléculas activas e inactivas.

In [ ]:
import seaborn as sns
sns.kdeplot(aux_data.logP[aux_data.activity==1], color="red")
sns.kdeplot(aux_data.logP[aux_data.activity==0])

Aquí vemos una tendencia clara. A mayor LogP, la molécula es más activa.

Pruébalo tú mismo con los otros descriptores (`hb_donors`, `hb_acceptors`, `rotable_bonds`, `mw`, `mr`, `tpsa`). Además de LogP, ¿qué descriptores tienen una distribución diferente según la actividad?

# Ejercicio Práctico

Ya aprendiste sobre los fingerprints como representaciones moleculares. Como son fáciles de calcular y siempre tienen una longitud fija, son muy adecuados como entrada para modelos de ML. Sin embargo, los fingerprints no son tan fáciles de interpretar para los humanos.

Tu tarea será entrenar un Random Forest con fingerprints de Morgan como input. Los fingerprints de Morgan ya han sido calculados para ti con la función `get_fingerprints(data)`.

In [ ]:
fps = get_fingerprints(data)
fps["activity"] = data.activity
fps.head()

`fps` contiene un total de 2049 columnas. 2048 de ellas son los bits respectivos del fingerprint. La última columna contiene la `activity`.

Primero, el conjunto de datos se divide en `training` (entrenamiento) y `test` (prueba). El 80% de los datos deben estar en el conjunto de entrenamiento y el 20% en el conjunto de prueba.

In [ ]:
train, test = train_test_split(_____,test_size= ___ , train_size= ______, random_state=1234)

train_x = __________
train_y = __________
test_x = __________
test_y = __________ 

Después de dividir los datos, entrena un clasificador Random Forest con el conjunto de entrenamiento.
Luego usa el modelo entrenado para clasificar las moléculas del conjunto de `test`.

In [ ]:
rf = RandomForestClassifier(n_estimators = 1000, random_state = 42)
rf.fit(_____, _____)
y_hat=rf.predict_proba(______)[:,1]

In [ ]:
roc_auc_score(___,____)

También podemos echar otro vistazo a la importancia de features:

In [ ]:
feat_importances = pd.Series(rf.feature_importances_, index=range(2048))
feat_importances.nlargest(20).nsmallest(20).plot(kind='barh', title = "Importance of Features")

Desafortunadamente, este gráfico ya no se puede interpretar con tanta facilidad, aunque queda claro que los cinco primeros bits son importantes para la predicción de actividad.

Los bits no se pueden graficar tan bien, pero con RDKit podemos mostrar las subestructuras que corresponden a cada bit.

In [ ]:
most_important_bits = feat_importances.nlargest(20).index.values
print("The 20 most important bits:", most_important_bits)
mol_ll = []
bi_ll = []


for i in range(20):
    bit = most_important_bits[i]
    for x in data.smiles:
        bi ={}
        mol = Chem.MolFromSmiles(x)
        fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, bitInfo=bi)
        if np.sum(np.array(list(bi))==bit)>0:
            mol_ll.append(mol)
            bi_ll.append(bi)
            break
        
prints=[(mol_ll[i],most_important_bits[i], bi_ll[i]) for i in range(20)]

Draw.DrawMorganBits(prints, useSVG=True, molsPerRow=3, legends= [str(most_important_bits[i]) for i in range(20)], subImgSize= [300,300])

El bit más importante para nuestros datos es un grupo hidroxilo fenólico (los átomos aromáticos están resaltados en amarillo, el átomo central en azul). Muchos otros fragmentos aromáticos también están representados entre los bits más importantes.